# 2.13 在线学习 (Online Learning)

模型在持续到来的数据流上逐步学习更新，是推荐系统、欺诈检测、内容审核等工业场景的核心范式。

本节涵盖：
- 增量学习与经验回放
- 概念漂移检测
- 在线梯度下降（OGD/FTRL）
- 自适应学习率
- 缓冲区管理策略
- 工业实践

## 1. 增量学习与经验回放

**目的**：在数据流上逐步更新模型，同时缓解灾难性遗忘

**基本原理**：在线学习每次只接收一个小批量数据，更新后丢弃。关键挑战是灾难性遗忘——模型在学习新数据时忘记旧知识。经验回放（Experience Replay）通过保留历史数据子集，在每次更新时混合新旧数据进行训练。

**核心概念**：
- **稳定性-可塑性困境**：可塑性使模型快速适应新数据，稳定性使模型保留旧知识
- **经验回放**：从缓冲区采样旧数据与新数据混合训练
- **回放比例**：新数据与旧数据的比例是关键超参数，通常0.3-0.7

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random

torch.manual_seed(42)

class OnlineLearner:
    def __init__(self, model, lr=1e-3, replay_size=100, replay_ratio=0.5):
        self.model = model
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.replay_buffer = deque(maxlen=replay_size)
        self.replay_ratio = replay_ratio
        self.step_count = 0

    def learn_step(self, X, y, use_replay=True):
        self.step_count += 1
        current_loss = F.cross_entropy(self.model(X), y)
        total_loss = current_loss

        if use_replay and len(self.replay_buffer) > 0:
            n_replay = max(1, int(len(X) * self.replay_ratio))
            replay_batch = random.sample(list(self.replay_buffer), min(n_replay, len(self.replay_buffer)))
            X_replay = torch.cat([b[0] for b in replay_batch])
            y_replay = torch.cat([b[1] for b in replay_batch])
            replay_loss = F.cross_entropy(self.model(X_replay), y_replay)
            total_loss = current_loss + replay_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        self.replay_buffer.append((X.detach(), y.detach()))
        return current_loss.item()

model_replay = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
model_no_replay = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
model_no_replay.load_state_dict(model_replay.state_dict())

learner_replay = OnlineLearner(model_replay, replay_size=50, replay_ratio=0.5)
learner_no = OnlineLearner(model_no_replay, replay_size=0)

print('=== Online Learning: With vs Without Experience Replay ===')
n_streams = 30
task1_acc_replay, task1_acc_no = [], []

for stream_idx in range(n_streams):
    if stream_idx < 15:
        X_stream = torch.randn(8, 10) + 1.0
        y_stream = torch.zeros(8, dtype=torch.long)
    else:
        X_stream = torch.randn(8, 10) - 1.0
        y_stream = torch.ones(8, dtype=torch.long)

    learner_replay.learn_step(X_stream, y_stream, use_replay=True)
    learner_no.learn_step(X_stream, y_stream, use_replay=False)

    if (stream_idx + 1) % 5 == 0:
        X_task1 = torch.randn(32, 10) + 1.0
        y_task1 = torch.zeros(32, dtype=torch.long)
        acc_r = (model_replay(X_task1).argmax(-1) == y_task1).float().mean()
        acc_n = (model_no_replay(X_task1).argmax(-1) == y_task1).float().mean()
        task1_acc_replay.append(acc_r.item())
        task1_acc_no.append(acc_n.item())
        phase = 'Task1' if stream_idx < 15 else 'Task2'
        print(f'Stream {stream_idx+1} [{phase}]: Task1_acc replay={acc_r:.3f} no_replay={acc_n:.3f}')

print(f'\nTask1 accuracy after drift:')
print(f'  With replay:    {task1_acc_replay[-1]:.3f}')
print(f'  Without replay: {task1_acc_no[-1]:.3f}')
print(f'\nKey: Experience replay preserves knowledge of earlier tasks.')
print(f'Without replay, the model catastrophically forgets Task1 when learning Task2.')

## 2. 概念漂移检测

**目的**：检测数据分布随时间的变化，触发模型更新策略

**基本原理**：在在线学习场景中，数据分布可能随时间变化（概念漂移），模型需要检测到这种变化并做出响应。常见检测方法：

- **DDM（Drift Detection Method）**：监控错误率的变化，当错误率显著上升时触发漂移警报
- **ADWIN（ADaptive WINdowing）**：动态调整窗口大小，检测窗口内统计量的显著变化
- **Page-Hinkley检验**：检测序列均值的变化

**工业场景**：
- 推荐系统：用户兴趣随季节/事件变化
- 欺诈检测：欺诈手段不断进化
- 内容审核：新类型的违规内容出现

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class DDMDetector:
    def __init__(self, min_samples=10, warning_level=2.0, drift_level=3.0):
        self.min_samples = min_samples
        self.warning_level = warning_level
        self.drift_level = drift_level
        self.n = 0
        self.p = 0.0
        self.s = 0.0
        self.p_min = float('inf')
        self.s_min = float('inf')

    def update(self, error):
        self.n += 1
        self.p = self.p + (error - self.p) / self.n
        self.s = math.sqrt(self.p * (1 - self.p) / self.n) if self.n > 1 else 0.0

        if self.n >= self.min_samples:
            if self.p + self.s < self.p_min + self.s_min:
                self.p_min = self.p
                self.s_min = self.s

            if self.p + self.s > self.p_min + self.warning_level * self.s_min:
                if self.p + self.s > self.p_min + self.drift_level * self.s_min:
                    return 'drift'
                return 'warning'
        return 'stable'

class ADWINDetector:
    def __init__(self, delta=0.002):
        self.delta = delta
        self.window = []
        self.total = 0.0
        self.total_sq = 0.0
        self.n = 0

    def update(self, value):
        self.window.append(value)
        self.total += value
        self.total_sq += value * value
        self.n += 1

        if self.n < 10:
            return 'stable'

        drift_detected = False
        for split in range(max(1, self.n - 100), self.n - 1):
            n0 = split
            n1 = self.n - split
            if n0 < 5 or n1 < 5:
                continue
            u0 = sum(self.window[:split]) / n0
            u1 = sum(self.window[split:]) / n1
            m = 1.0 / (1.0/n0 + 1.0/n1)
            eps = math.sqrt(0.5 * math.log(2.0 / self.delta) / m)
            if abs(u0 - u1) >= eps:
                drift_detected = True
                self.window = self.window[split:]
                self.n = len(self.window)
                self.total = sum(self.window)
                self.total_sq = sum(v*v for v in self.window)
                break

        return 'drift' if drift_detected else 'stable'

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ddm = DDMDetector()
adwin = ADWINDetector()

print('=== Concept Drift Detection ===')
drift_point = 50
for step in range(100):
    if step < drift_point:
        X = torch.randn(16, 10) + 2.0
        y = torch.zeros(16, dtype=torch.long)
    else:
        X = torch.randn(16, 10) - 2.0
        y = torch.ones(16, dtype=torch.long)

    with torch.no_grad():
        preds = model(X).argmax(-1)
        error_rate = (preds != y).float().mean().item()

    loss = F.cross_entropy(model(X), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ddm_status = ddm.update(error_rate)
    adwin_status = adwin.update(error_rate)

    if ddm_status == 'drift' or adwin_status == 'drift':
        print(f'Step {step}: DDM={ddm_status}, ADWIN={adwin_status}, error_rate={error_rate:.3f}')
        if step > drift_point:
            break

print(f'\nTrue drift at step {drift_point}')
print(f'DDM detected drift at step {ddm.n}')
print(f'\nKey: DDM monitors error rate increase; ADWIN detects distribution change in sliding window.')
print(f'In production, drift detection triggers model retraining or adaptation.')

## 3. 在线梯度下降（OGD/FTRL）

**目的**：为在线学习设计高效、有理论保证的优化算法

**OGD（Online Gradient Descent）**：最简单的在线学习算法，每步接收数据、计算梯度、更新参数。遗憾界（regret bound）为O(√T)。

**FTRL（Follow The Regularized Leader）**：OGD的改进版本，在每步选择使历史累积损失最小的参数，同时加入正则化防止参数变化过大。FTRL是工业推荐系统（如Google的广告系统）的核心优化算法。

**FTRL vs OGD**：
- OGD：θ_{t+1} = θ_t - η·∇f_t(θ_t)，简单但可能震荡
- FTRL：θ_{t+1} = argmin_θ {Σ_{i=1}^t f_i(θ) + λ₁|θ|₁ + λ₂/2·||θ||₂²}，更稳定
- FTRL的L1正则化天然产生稀疏解，适合高维特征场景

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class FTRLProximal:
    def __init__(self, d, alpha=0.1, beta=1.0, l1=1.0, l2=1.0):
        self.alpha = alpha
        self.beta = beta
        self.l1 = l1
        self.l2 = l2
        self.z = torch.zeros(d)
        self.n = torch.zeros(d)
        self.w = torch.zeros(d)

    def predict(self, x):
        return (x * self.w).sum()

    def update(self, x, grad):
        sigma = (torch.sqrt(self.n + grad ** 2) - torch.sqrt(self.n)) / self.alpha
        self.z += grad - sigma * self.w
        self.n += grad ** 2

        for i in range(len(self.w)):
            if abs(self.z[i]) <= self.l1:
                self.w[i] = 0.0
            else:
                sign = 1.0 if self.z[i] > 0 else -1.0
                self.w[i] = -(self.z[i] - sign * self.l1) / ((self.beta + torch.sqrt(self.n[i])) / self.alpha + self.l2)

n_features = 20
n_steps = 200
true_w = torch.randn(n_features)
true_w[10:] = 0

ftrl = FTRLProximal(n_features, alpha=0.1, l1=1.0, l2=1.0)
ogd_w = torch.zeros(n_features, requires_grad=True)
ogd_lr = 0.01

ftrl_losses, ogd_losses = [], []
ftrl_sparsity, ogd_sparsity = [], []

print('=== OGD vs FTRL Comparison ===')
for step in range(n_steps):
    x = torch.randn(n_features)
    y = (x * true_w).sum() + torch.randn(1) * 0.1

    ftrl_pred = ftrl.predict(x)
    ftrl_loss = (ftrl_pred - y) ** 2
    ftrl_grad = 2 * (ftrl_pred - y) * x
    ftrl.update(x, ftrl_grad.detach())
    ftrl_losses.append(ftrl_loss.item())

    ogd_pred = (x * ogd_w).sum()
    ogd_loss = (ogd_pred - y) ** 2
    ogd_loss.backward()
    with torch.no_grad():
        ogd_w -= ogd_lr * ogd_w.grad
        ogd_w.grad = None
    ogd_losses.append(ogd_loss.item())

    if (step + 1) % 50 == 0:
        ftrl_nz = (ftrl.w.abs() > 1e-6).sum().item()
        ogd_nz = (ogd_w.abs() > 1e-6).sum().item()
        ftrl_sparsity.append(ftrl_nz)
        ogd_sparsity.append(ogd_nz)
        print(f'Step {step+1}: FTRL_loss={sum(ftrl_losses[-50:])/50:.4f}, '
              f'OGD_loss={sum(ogd_losses[-50:])/50:.4f}, '
              f'FTRL_nz={ftrl_nz}/{n_features}, OGD_nz={ogd_nz}/{n_features}')

print(f'\nTrue non-zero features: {(true_w.abs() > 1e-6).sum().item()}/{n_features}')
print(f'FTRL sparsity: {(ftrl.w.abs() < 1e-6).sum().item()}/{n_features} zeros (L1 regularization)')
print(f'OGD sparsity:  {(ogd_w.abs() < 1e-6).sum().item()}/{n_features} zeros')
print(f'\nKey: FTRL with L1 produces sparse solutions, critical for high-dimensional industrial systems.')
print(f'Google uses FTRL for ad CTR prediction with billions of features.')

## 4. 自适应学习率与缓冲区管理

**自适应学习率**：在线学习中，数据分布可能变化，固定学习率难以适应。常用策略：
- **衰减学习率**：η_t = η₀ / (1 + λt)，保证收敛但可能适应过慢
- **检测驱动调整**：检测到漂移时增大学习率，稳定后衰减
- **元学习率**：根据验证集表现动态调整

**缓冲区管理策略**：
- **FIFO**：先进先出，简单但可能丢失重要旧数据
- **水库采样（Reservoir Sampling）**：每个样本被保留的概率相等，保证无偏
- **优先级回放**：按TD-error或损失大小排序，优先回放"难"样本
- **类平衡回放**：确保缓冲区中各类别比例均衡

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import math

torch.manual_seed(42)

class ReservoirBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.n_seen = 0

    def add(self, item):
        self.n_seen += 1
        if len(self.buffer) < self.capacity:
            self.buffer.append(item)
        else:
            j = random.randint(0, self.n_seen - 1)
            if j < self.capacity:
                self.buffer[j] = item

    def sample(self, n):
        n = min(n, len(self.buffer))
        return random.sample(self.buffer, n)

class PriorityBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.priorities = []

    def add(self, item, priority=1.0):
        if len(self.buffer) >= self.capacity:
            min_idx = self.priorities.index(min(self.priorities))
            if priority > self.priorities[min_idx]:
                self.buffer[min_idx] = item
                self.priorities[min_idx] = priority
        else:
            self.buffer.append(item)
            self.priorities.append(priority)

    def update_priority(self, idx, priority):
        if idx < len(self.priorities):
            self.priorities[idx] = priority

    def sample(self, n, beta=0.4):
        n = min(n, len(self.buffer))
        probs = torch.tensor(self.priorities) ** self.alpha
        probs = probs / probs.sum()
        indices = torch.multinomial(probs, n, replacement=False)
        weights = (len(self.buffer) * probs[indices]) ** (-beta)
        weights = weights / weights.max()
        return [self.buffer[i] for i in indices.tolist()], weights, indices

class AdaptiveOnlineLearner:
    def __init__(self, model, base_lr=1e-3, buffer_type='reservoir', buffer_size=100):
        self.model = model
        self.base_lr = base_lr
        self.current_lr = base_lr
        self.optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
        self.step_count = 0
        self.drift_detected = False
        self.loss_history = []

        if buffer_type == 'reservoir':
            self.buffer = ReservoirBuffer(buffer_size)
        else:
            self.buffer = PriorityBuffer(buffer_size)
        self.buffer_type = buffer_type

    def adjust_lr_on_drift(self, drift_detected):
        if drift_detected:
            self.current_lr = self.base_lr * 5.0
            self.drift_detected = True
        elif self.drift_detected:
            self.current_lr = max(self.base_lr, self.current_lr * 0.9)
            if self.current_lr < self.base_lr * 1.1:
                self.current_lr = self.base_lr
                self.drift_detected = False

        for pg in self.optimizer.param_groups:
            pg['lr'] = self.current_lr

    def learn_step(self, X, y, drift_detected=False):
        self.step_count += 1
        self.adjust_lr_on_drift(drift_detected)

        logits = self.model(X)
        loss = F.cross_entropy(logits, y)

        if len(self.buffer.buffer) > 0:
            if self.buffer_type == 'reservoir':
                replay_data = self.buffer.sample(min(4, len(self.buffer.buffer)))
                X_r = torch.cat([d[0] for d in replay_data])
                y_r = torch.cat([d[1] for d in replay_data])
            else:
                replay_data, weights, _ = self.buffer.sample(min(4, len(self.buffer.buffer)))
                X_r = torch.cat([d[0] for d in replay_data])
                y_r = torch.cat([d[1] for d in replay_data])
            replay_loss = F.cross_entropy(self.model(X_r), y_r)
            loss = loss + replay_loss

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        self.buffer.add((X.detach(), y.detach()), priority=loss.item())
        self.loss_history.append(loss.item())
        return loss.item()

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
learner = AdaptiveOnlineLearner(model, base_lr=1e-3, buffer_type='reservoir', buffer_size=50)

print('=== Adaptive Online Learning with Drift Response ===')
drift_step = 30
for step in range(60):
    if step < drift_step:
        X = torch.randn(8, 10) + 2.0
        y = torch.zeros(8, dtype=torch.long)
        drift = False
    else:
        X = torch.randn(8, 10) - 2.0
        y = torch.ones(8, dtype=torch.long)
        drift = (step == drift_step)

    loss = learner.learn_step(X, y, drift_detected=drift)

    if (step + 1) % 10 == 0:
        print(f'Step {step+1}: loss={loss:.4f}, lr={learner.current_lr:.6f}, '
              f'buffer_size={len(learner.buffer.buffer)}, drift={learner.drift_detected}')

print(f'\nKey: On drift detection, learning rate spikes to quickly adapt,')
print(f'then gradually decays back to stable level.')
print(f'Reservoir sampling ensures unbiased representation of historical data.')

## 5. 工业实践：流式训练架构

**目的**：构建可扩展的在线学习生产系统

**流式训练架构核心组件**：

1. **数据采集层**：实时收集用户行为日志（点击、曝光、停留时长等）
2. **特征工程层**：实时特征提取和交叉，通常使用特征中心（Feature Store）
3. **训练层**：增量模型更新，支持A/B实验和模型回滚
4. **服务层**：模型推理和在线预测，要求低延迟（<50ms）

**关键工程决策**：
- **更新频率**：实时（秒级）vs 准实时（分钟级）vs 周期（小时级）
- **模型版本管理**：支持快速回滚到上一版本
- **特征一致性**：训练和推理使用相同的特征计算逻辑
- **数据质量监控**：检测数据延迟、缺失、异常

**典型系统**：
- **推荐系统**：淘宝/字节跳动使用分钟级增量更新
- **搜索排序**：Google使用FTRL在线更新排序模型
- **广告竞价**：实时CTR模型更新，延迟要求<10ms

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from collections import deque
import random

torch.manual_seed(42)

class StreamingTrainingSystem:
    def __init__(self, model, update_interval=5, buffer_size=200, lr=1e-3):
        self.model = model
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.data_buffer = deque(maxlen=buffer_size)
        self.update_interval = update_interval
        self.model_version = 0
        self.model_checkpoints = []
        self.metrics_history = deque(maxlen=100)
        self.n_updates = 0

    def ingest(self, X, y):
        self.data_buffer.append((X, y))

    def should_update(self):
        return len(self.data_buffer) >= self.update_interval

    def train_step(self):
        if not self.should_update():
            return None

        batch = random.sample(list(self.data_buffer), min(self.update_interval, len(self.data_buffer)))
        X = torch.cat([b[0] for b in batch])
        y = torch.cat([b[1] for b in batch])

        logits = self.model(X)
        loss = F.cross_entropy(logits, y)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        self.n_updates += 1
        acc = (logits.argmax(-1) == y).float().mean().item()
        self.metrics_history.append({'loss': loss.item(), 'acc': acc, 'version': self.model_version})

        if self.n_updates % 10 == 0:
            self.model_version += 1
            self.model_checkpoints.append(
                {k: v.clone() for k, v in self.model.state_dict().items()}
            )
            if len(self.model_checkpoints) > 3:
                self.model_checkpoints.pop(0)

        return {'loss': loss.item(), 'acc': acc}

    def rollback(self):
        if self.model_checkpoints:
            self.model.load_state_dict(self.model_checkpoints[-1])
            self.model_version -= 1
            return True
        return False

    def evaluate(self, X, y):
        with torch.no_grad():
            logits = self.model(X)
            loss = F.cross_entropy(logits, y)
            acc = (logits.argmax(-1) == y).float().mean()
        return {'loss': loss.item(), 'acc': acc.item()}

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
system = StreamingTrainingSystem(model, update_interval=4, buffer_size=100, lr=1e-3)

print('=== Streaming Training System Simulation ===')
for step in range(50):
    X = torch.randn(4, 10) + (1.0 if step < 25 else -1.0)
    y = torch.tensor([0]*4 if step < 25 else [1]*4, dtype=torch.long)

    system.ingest(X, y)
    result = system.train_step()

    if result and (step + 1) % 10 == 0:
        X_eval = torch.randn(32, 10) + (1.0 if step < 25 else -1.0)
        y_eval = torch.tensor([0]*32 if step < 25 else [1]*32, dtype=torch.long)
        eval_result = system.evaluate(X_eval, y_eval)
        print(f'Step {step+1}: train_loss={result["loss"]:.4f}, '
              f'eval_acc={eval_result["acc"]:.3f}, '
              f'version=v{system.model_version}, '
              f'checkpoints={len(system.model_checkpoints)}, '
              f'buffer={len(system.data_buffer)}')

print(f'\nSystem stats:')
print(f'  Total updates: {system.n_updates}')
print(f'  Model versions: {system.model_version}')
print(f'  Checkpoints available for rollback: {len(system.model_checkpoints)}')
print(f'\nKey: Production online learning systems need:')
print(f'  1. Buffered data ingestion for stable training')
print(f'  2. Periodic checkpointing for rollback capability')
print(f'  3. Continuous evaluation to detect degradation')
print(f'  4. Version management for A/B testing and safe deployment')